# Notebook to prepare the hexagons for the analysis

Creates the hexagons in resolution 6 and 7. 

## Set wd
Choose your absolute path to the Lithium folder

In [4]:
import os

# Global working directory for this notebook session
os.chdir("/Users/jakobnitschke/Documents/GitHub/Lithium")

print("cwd:", os.getcwd())

cwd: /Users/jakobnitschke/Documents/GitHub/Lithium


## Resolution 6 Hexagons

- **Input:** `Data/Endorheic_basins_Puna.geojson` (basin polygons with `CUENCA`)
- **Output (hexagons GeoJSON):** `Data/lithium_hexagons_puna_res6.geojson` (H3 res=6 hex grid + `CUENCA` membership)
- **Output (interactive map HTML):** `Data/lithium_hexagons_puna_visualization.html` (Folium map of basins + hexagons)

- Converts basin polygons (`CUENCA`) into an **H3 hexagon grid (resolution 6)**.
- Assigns each hexagon to the basin(s) it overlaps.
- Exports the hexagons as a **GeoJSON**.
- Creates an **interactive Folium map** to visually check basin–hexagon alignment.

In [12]:
import h3
from h3.api.basic_str import LatLngPoly  # <-- new
import geopandas as gpd
from shapely.geometry import Polygon as ShapelyPolygon
import folium
import pandas as pd

def get_hexagons_for_geojson(geojson_path, resolution):
    # Read the GeoJSON file
    gdf = gpd.read_file(geojson_path)

    # Dictionary to track which polygon(s) each hexagon belongs to
    hexagon_to_cuenca = {}

    # Process each polygon in the GeoJSON
    for idx, row in gdf.iterrows():
        geom = row.geometry
        cuenca_name = row['CUENCA']

        # Handle different geometry types
        if geom.geom_type == 'MultiPolygon':
            polygons = list(geom.geoms)
        else:
            polygons = [geom]

        # Process each polygon
        for polygon in polygons:
            if hasattr(polygon, 'exterior'):
                # Get coordinates and handle 3D coordinates if present
                if len(polygon.exterior.coords[0]) > 2:
                    coords = [(x, y) for x, y, *_ in polygon.exterior.coords]
                else:
                    coords = list(polygon.exterior.coords)

                # Convert to (lat, lng) format for h3
                vertices = [(y, x) for x, y in coords]

                # Create h3 polygon and get cells
                h3_polygon = LatLngPoly(vertices)  # vertices are (lat, lng)
                hexagons = h3.polygon_to_cells(h3_polygon, resolution)

                # Associate each hexagon with the current CUENCA
                for hex_id in hexagons:
                    if hex_id in hexagon_to_cuenca:
                        if cuenca_name not in hexagon_to_cuenca[hex_id]:
                            hexagon_to_cuenca[hex_id].append(cuenca_name)
                    else:
                        hexagon_to_cuenca[hex_id] = [cuenca_name]

    # Get all unique hexagons
    all_hexagons = list(hexagon_to_cuenca.keys())

    # Create geometries for each hexagon
    geometries = [ShapelyPolygon([(lng, lat) for lat, lng in h3.cell_to_boundary(h)]) for h in all_hexagons]

    # Create DataFrame with hexagon indices and their CUENCA values
    df_data = {
        'h3_index': all_hexagons,
        'CUENCA': [';'.join(hexagon_to_cuenca[h]) for h in all_hexagons]
    }

    # Create GeoDataFrame
    hexagon_gdf = gpd.GeoDataFrame(
        data=df_data,
        geometry=geometries,
        crs="EPSG:4326"
    )

    return hexagon_gdf

# Path to your GeoJSON file
geojson_path = "Data/Endorheic_basins_Puna.geojson"

# Create GeoDataFrame with hexagons
resolution = 6  # You can adjust this based on your needs
hexagon_gdf = get_hexagons_for_geojson(geojson_path, resolution)

print(f"Created {len(hexagon_gdf)} hexagons at resolution {resolution}")
print(f"Hexagons by CUENCA: {hexagon_gdf['CUENCA'].value_counts().to_dict()}")
hexagon_gdf.head()

# Optional: Save hexagons to a new GeoJSON file
output_path = "Data/lithium_hexagons_puna_res6.geojson"
hexagon_gdf.to_file(output_path, driver="GeoJSON")

# Optional: Visualize the hexagons colored by CUENCA
def visualize_hexagons_by_cuenca(original_gdf, hexagon_gdf):
    # Create a map centered on the original data
    center = original_gdf.unary_union.centroid
    m = folium.Map(location=[center.y, center.x], zoom_start=8)

    # Add the original polygons
    folium.GeoJson(
        original_gdf,
        name='Original Basins',
        style_function=lambda x: {
            'fillColor': 'blue',
            'color': 'blue',
            'weight': 2,
            'fillOpacity': 0.1
        },
        tooltip=folium.GeoJsonTooltip(fields=['CUENCA'], aliases=['Basin:'])
    ).add_to(m)

    # Get unique CUENCA values for coloring
    unique_cuencas = set()
    for cuenca_str in hexagon_gdf['CUENCA']:
        for cuenca in cuenca_str.split(';'):
            unique_cuencas.add(cuenca)

    # Create color map
    import random
    color_map = {cuenca: f"#{random.randint(0, 0xFFFFFF):06x}" for cuenca in unique_cuencas}

    # Function to determine color based on CUENCA value
    def get_color(cuenca_str):
        cuencas = cuenca_str.split(';')
        if len(cuencas) == 1:
            return color_map[cuencas[0]]
        else:
            # For hexagons in multiple basins, use a distinct color
            return 'purple'

    # Add the hexagons colored by CUENCA
    for idx, row in hexagon_gdf.iterrows():
        folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x, cuenca=row['CUENCA']: {
                'fillColor': get_color(cuenca),
                'color': 'black',
                'weight': 0.5,
                'fillOpacity': 0.5
            },
            tooltip=row['CUENCA']
        ).add_to(m)

    # Add a legend
    from branca.element import Template, MacroElement

    template = """
    {% macro html(this, kwargs) %}
    <div style="position: fixed; bottom: 50px; left: 50px; width: 150px; height: auto; z-index:9999; font-size:14px; background-color: white; padding: 10px; border: 2px solid grey;">
    <h4>Legend</h4>
    {% for cuenca in cuencas %}
    <div>
      <span style="background-color: {{colors[cuenca]}}; display: inline-block; width: 12px; height: 12px;"></span>
      <span>{{cuenca}}</span>
    </div>
    {% endfor %}
    <div>
      <span style="background-color: purple; display: inline-block; width: 12px; height: 12px;"></span>
      <span>Multiple basins</span>
    </div>
    </div>
    {% endmacro %}
    """

    macro = MacroElement()
    macro._template = Template(template)
    macro.cuencas = list(unique_cuencas)
    macro.colors = color_map
    #m.get_root().add_child(macro)

    # Add layer control
    #folium.LayerControl().add_to(m)

    return m

# Visualize
original_gdf = gpd.read_file(geojson_path)
m = visualize_hexagons_by_cuenca(original_gdf, hexagon_gdf)
m.save("Data/lithium_hexagons_puna_visualization.html")


Created 2649 hexagons at resolution 6
Hexagons by CUENCA: {'Laguna de Guayatayoc': 341, 'Salar de Arizaro': 202, 'Salar Carachi Pampa': 190, 'Salar de Antofalla': 160, 'Cuencas endorreicas en Catamarca (E)': 151, 'Salinas Grandes de Jujuy y Salta': 138, 'Salar del Hombre Muerto': 113, 'Laguna de Pozuelos': 108, 'Laguna Blanca': 106, 'Cuencas endorreicas en Catamarca (H)': 89, 'Salar de Olaroz': 89, 'Cuencas endorreicas en Salta (A)': 75, 'Salar de Pocitos o Quirón': 73, 'Salar de Cauchari': 72, 'Salar de Rincón': 63, 'Salar de Pastos Grandes': 50, 'Salar de Llullaillaco': 49, 'Cuencas endorreicas en Catamarca (B)': 48, 'Salar del Fraile': 47, 'Laguna Verde': 45, 'Salar de Río Grande': 41, 'Laguna de las Salinas o Tres Quebradas': 35, 'Salar de Jama': 34, 'Salar de Incahuasi en Catamarca': 34, 'Cuencas endorreicas en Salta (F)': 27, 'Cuencas endorreicas en Salta (B)': 26, 'Cuencas endorreicas en Catamarca (D)': 26, 'Cuencas endorreicas en Salta y Catamarca (A)': 20, 'Cuencas endorreicas

/var/folders/96/ccn1qht122b11lrdgngbr2600000gn/T/ipykernel_17320/2255403557.py:89: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = original_gdf.unary_union.centroid


In [14]:
hexagon_gdf.columns
hexagon_gdf["h3_index"].head()

0    86b22c677ffffff
1    86b22c84fffffff
2    86b228c47ffffff
3    86b22856fffffff
4    86b228ccfffffff
Name: h3_index, dtype: object

In [ ]:
## Resolution 7 Hexagons

- **Input:** `Data/Endorheic_basins_Puna.geojson` (basin polygons with `CUENCA`)
- **Output (hexagons GeoJSON):** `Data/lithium_hexagons_puna_res7.geojson` (H3 res=7 hex grid + `CUENCA` membership)
- **Output (interactive map HTML):** `Data/lithium_hexagons_puna_visualization.html` (Folium map of basins + hexagons)

- Converts basin polygons (`CUENCA`) into an **H3 hexagon grid (resolution 7)**.
- Assigns each hexagon to the basin(s) it overlaps.
- Exports the hexagons as a **GeoJSON**.
- Creates an **interactive Folium map** to visually check basin–hexagon alignment.

In [15]:
import h3
from h3.api.basic_str import LatLngPoly  # <-- new
import geopandas as gpd
from shapely.geometry import Polygon as ShapelyPolygon
import folium
import pandas as pd

def get_hexagons_for_geojson(geojson_path, resolution):
    # Read the GeoJSON file
    gdf = gpd.read_file(geojson_path)

    # Dictionary to track which polygon(s) each hexagon belongs to
    hexagon_to_cuenca = {}

    # Process each polygon in the GeoJSON
    for idx, row in gdf.iterrows():
        geom = row.geometry
        cuenca_name = row['CUENCA']

        # Handle different geometry types
        if geom.geom_type == 'MultiPolygon':
            polygons = list(geom.geoms)
        else:
            polygons = [geom]

        # Process each polygon
        for polygon in polygons:
            if hasattr(polygon, 'exterior'):
                # Get coordinates and handle 3D coordinates if present
                if len(polygon.exterior.coords[0]) > 2:
                    coords = [(x, y) for x, y, *_ in polygon.exterior.coords]
                else:
                    coords = list(polygon.exterior.coords)

                # Convert to (lat, lng) format for h3
                vertices = [(y, x) for x, y in coords]

                # Create h3 polygon and get cells
                h3_polygon = LatLngPoly(vertices)  # vertices are (lat, lng)
                hexagons = h3.polygon_to_cells(h3_polygon, resolution)

                # Associate each hexagon with the current CUENCA
                for hex_id in hexagons:
                    if hex_id in hexagon_to_cuenca:
                        if cuenca_name not in hexagon_to_cuenca[hex_id]:
                            hexagon_to_cuenca[hex_id].append(cuenca_name)
                    else:
                        hexagon_to_cuenca[hex_id] = [cuenca_name]

    # Get all unique hexagons
    all_hexagons = list(hexagon_to_cuenca.keys())

    # Create geometries for each hexagon
    geometries = [ShapelyPolygon([(lng, lat) for lat, lng in h3.cell_to_boundary(h)]) for h in all_hexagons]

    # Create DataFrame with hexagon indices and their CUENCA values
    df_data = {
        'h3_index': all_hexagons,
        'CUENCA': [';'.join(hexagon_to_cuenca[h]) for h in all_hexagons]
    }

    # Create GeoDataFrame
    hexagon_gdf = gpd.GeoDataFrame(
        data=df_data,
        geometry=geometries,
        crs="EPSG:4326"
    )

    return hexagon_gdf

# Path to your GeoJSON file
geojson_path = "Data/Endorheic_basins_Puna.geojson"

# Create GeoDataFrame with hexagons
resolution = 7  # You can adjust this based on your needs
hexagon_gdf = get_hexagons_for_geojson(geojson_path, resolution)

print(f"Created {len(hexagon_gdf)} hexagons at resolution {resolution}")
print(f"Hexagons by CUENCA: {hexagon_gdf['CUENCA'].value_counts().to_dict()}")
hexagon_gdf.head()

# Optional: Save hexagons to a new GeoJSON file
output_path = "Data/lithium_hexagons_puna_res7.geojson"
hexagon_gdf.to_file(output_path, driver="GeoJSON")

# Optional: Visualize the hexagons colored by CUENCA
def visualize_hexagons_by_cuenca(original_gdf, hexagon_gdf):
    # Create a map centered on the original data
    center = original_gdf.unary_union.centroid
    m = folium.Map(location=[center.y, center.x], zoom_start=8)

    # Add the original polygons
    folium.GeoJson(
        original_gdf,
        name='Original Basins',
        style_function=lambda x: {
            'fillColor': 'blue',
            'color': 'blue',
            'weight': 2,
            'fillOpacity': 0.1
        },
        tooltip=folium.GeoJsonTooltip(fields=['CUENCA'], aliases=['Basin:'])
    ).add_to(m)

    # Get unique CUENCA values for coloring
    unique_cuencas = set()
    for cuenca_str in hexagon_gdf['CUENCA']:
        for cuenca in cuenca_str.split(';'):
            unique_cuencas.add(cuenca)

    # Create color map
    import random
    color_map = {cuenca: f"#{random.randint(0, 0xFFFFFF):06x}" for cuenca in unique_cuencas}

    # Function to determine color based on CUENCA value
    def get_color(cuenca_str):
        cuencas = cuenca_str.split(';')
        if len(cuencas) == 1:
            return color_map[cuencas[0]]
        else:
            # For hexagons in multiple basins, use a distinct color
            return 'purple'

    # Add the hexagons colored by CUENCA
    for idx, row in hexagon_gdf.iterrows():
        folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x, cuenca=row['CUENCA']: {
                'fillColor': get_color(cuenca),
                'color': 'black',
                'weight': 0.5,
                'fillOpacity': 0.5
            },
            tooltip=row['CUENCA']
        ).add_to(m)

    # Add a legend
    from branca.element import Template, MacroElement

    template = """
    {% macro html(this, kwargs) %}
    <div style="position: fixed; bottom: 50px; left: 50px; width: 150px; height: auto; z-index:9999; font-size:14px; background-color: white; padding: 10px; border: 2px solid grey;">
    <h4>Legend</h4>
    {% for cuenca in cuencas %}
    <div>
      <span style="background-color: {{colors[cuenca]}}; display: inline-block; width: 12px; height: 12px;"></span>
      <span>{{cuenca}}</span>
    </div>
    {% endfor %}
    <div>
      <span style="background-color: purple; display: inline-block; width: 12px; height: 12px;"></span>
      <span>Multiple basins</span>
    </div>
    </div>
    {% endmacro %}
    """

    macro = MacroElement()
    macro._template = Template(template)
    macro.cuencas = list(unique_cuencas)
    macro.colors = color_map
    #m.get_root().add_child(macro)

    # Add layer control
    #folium.LayerControl().add_to(m)

    return m

# Visualize
original_gdf = gpd.read_file(geojson_path)
m = visualize_hexagons_by_cuenca(original_gdf, hexagon_gdf)
m.save("Data/lithium_hexagons_puna_visualization.html")


Created 18588 hexagons at resolution 7
Hexagons by CUENCA: {'Laguna de Guayatayoc': 2399, 'Salar de Arizaro': 1400, 'Salar Carachi Pampa': 1325, 'Salar de Antofalla': 1118, 'Cuencas endorreicas en Catamarca (E)': 1069, 'Salinas Grandes de Jujuy y Salta': 966, 'Salar del Hombre Muerto': 793, 'Laguna de Pozuelos': 753, 'Laguna Blanca': 745, 'Cuencas endorreicas en Catamarca (H)': 621, 'Salar de Olaroz': 615, 'Cuencas endorreicas en Salta (A)': 517, 'Salar de Pocitos o Quirón': 506, 'Salar de Cauchari': 501, 'Salar de Rincón': 454, 'Cuencas endorreicas en Catamarca (B)': 348, 'Salar de Pastos Grandes': 344, 'Salar del Fraile': 343, 'Salar de Llullaillaco': 334, 'Laguna Verde': 326, 'Salar de Río Grande': 291, 'Laguna de las Salinas o Tres Quebradas': 242, 'Salar de Jama': 237, 'Salar de Incahuasi en Catamarca': 233, 'Cuencas endorreicas en Catamarca (D)': 188, 'Cuencas endorreicas en Salta (F)': 182, 'Cuencas endorreicas en Salta (B)': 168, 'Cuencas endorreicas en Salta y Catamarca (A)': 

/var/folders/96/ccn1qht122b11lrdgngbr2600000gn/T/ipykernel_17320/1782476473.py:89: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = original_gdf.unary_union.centroid
